# Binomial Option Pricer: live data or your own inputs

**What this notebook does**
1. Gets today's share price and price history (live from Yahoo Finance, or numbers you type in).
2. Estimates how much the stock moves (volatility).
3. Prices an option with a binomial tree and checks it against Black-Scholes.
4. Compares the model price with the real market price and finds the volatility the market is assuming (implied volatility).
5. Shows the Greeks, draws strategy payoff charts, and has an interactive explorer.

**How to use it:** edit only the CONFIG cell below, then choose *Run All*.
The full explanation (theory, process, troubleshooting) is in `README.md`.

*Educational tool. Not investment advice.*

## 1. CONFIG (the only cell you need to edit)

In [ ]:
# =====================================================================
#  CONFIG
# =====================================================================

# ---- A) WHAT DO YOU WANT TO PRICE? ------------------------------------
SYMBOL      = "RELIANCE"      # NSE symbol
YF_TICKER   = "RELIANCE.NS"   # Yahoo Finance ticker (NSE stocks end with .NS, BSE with .BO)
OPTION_TYPE = "call"          # "call" or "put"
STRIKE      = 1200.0          # strike price in rupees
EXPIRY_DATE = None            # "YYYY-MM-DD" in quotes. None = the nearest upcoming monthly expiry (last Tuesday of the month)
LOT_SIZE    = 500             # contract size. Check your broker, it can change
EXERCISE    = "european"      # NSE stock options are European. "american" also works
VALUATION_DATE = None         # "YYYY-MM-DD" to price as of a past day. None = today
EXCHANGE_HOLIDAYS = ["2026-10-02", "2026-10-20", "2026-11-10", "2026-11-24", "2026-12-25"]
#   NSE trading holidays on weekdays. Only used to move the AUTOMATIC expiry to the previous trading day.
#   This list was copied from a public holiday table (Oct-Dec 2026). Check it on nseindia.com and add later dates yourself.

# ---- B) WHERE DO THE NUMBERS COME FROM? -------------------------------
DATA_MODE = "auto"
#   "live"   -> download price + history from Yahoo Finance (needs internet and `pip install yfinance`)
#   "manual" -> use only what you type in MANUAL below (works offline)
#   "auto"   -> try live first; if that fails, fall back to MANUAL and tell you
#
# How each MANUAL field is used:
#   spot                 -> used when NOT live (manual mode, or auto after a failed download).
#                           Leave None to use the last close in your CSV.
#   price_csv            -> used when live history is not available. File with columns
#                           Date, Close (optional: High, Low, gap). Set None if you have none.
#   volatility           -> if set, ALWAYS overrides the estimate from history (even in live mode)
#   risk_free_rate       -> always used (there is no free live feed). UPDATE IT. See README.
#   dividend_yield       -> always used (continuous yearly yield, 0.0 for most Indian stocks)
#   market_option_price  -> last traded price of THIS option (optional, enables the cheap/expensive check)
#   market_bid, market_ask -> optional. If both are given, their middle is used (better than last price)
MANUAL = {
    "spot":                1202.00,
    "volatility":          None,       # decimal: 0.21 means 21% per year
    "risk_free_rate":      0.0557,     # decimal: 91-day T-bill yield, 5.57% (auction of 7 Oct 2026)
    "dividend_yield":      0.0,
    "price_csv":           "data/RELIANCE_sample_prices.csv",
    "market_option_price": None,
    "market_bid":          None,
    "market_ask":          None,
}

# ---- C) MODEL SETTINGS --------------------------------------------------
VOL_METHOD = "close"   # "close"     = close-to-close (standard)
                       # "ewma"      = recent days count more (RiskMetrics, lambda 0.94)
                       # "parkinson" = uses daily High/Low (needs High and Low columns)
                       # "implied"   = take the volatility from a real option's market price (see IV_REFERENCE below)
VOL_WINDOW = 63        # trading days of history (63 is about 3 months). Not used by "ewma"
TREE_STEPS = 500       # steps in the tree. More steps = more accurate but slower. 300 to 1000 is fine
# Used only when VOL_METHOD = "implied". The option whose market price gives the volatility.
# It must have the SAME EXPIRY as the option you are pricing. Give "price" (last or middle) OR both "bid" and "ask".
# Tip: use the at-the-money option. If you use the very option you are pricing, the model price just equals its market price.
IV_REFERENCE = {"option": "call", "strike": 1200.0, "price": None, "bid": None, "ask": None}
DIVIDENDS  = []        # known cash dividends before expiry, e.g. [("2026-12-01", 5.0)]  (ex-date, rupees per share)

# ---- D) COMPARING WITH THE MARKET ---------------------------------------
EDGE_BUFFER_PCT = 5.0    # ignore gaps smaller than this % of the model price (costs, noise)
FETCH_NSE_QUOTE = False  # EXPERIMENTAL: try to read this option's price from NSE's website

# ---- E) STRATEGY BUILDER --------------------------------------------------
STRATEGY       = "bull_call_spread"  # covered_call, protective_put, bull_call_spread,
                                     # bear_put_spread, long_straddle, long_strangle, custom
STRATEGY_WIDTH = 100                 # rupee gap between strikes for spreads / strangles
USE_MARKET_PRICE_FOR_MAIN_LEG = True # use the market price for the leg that matches OPTION_TYPE + STRIKE
# Used when STRATEGY = "custom". qty: +1 buy, -1 sell (in lots). "premium" is optional (rupees per share).
CUSTOM_LEGS = [
    {"kind": "call",  "strike": 1200, "qty": +1},
    {"kind": "put",   "strike": 1100, "qty": -1},
    # {"kind": "stock", "qty": +1},
]

## 2. Setup (nothing to edit below this line)

In [ ]:
import math
import os
import datetime as dt

import numpy as np
import pandas as pd
import matplotlib

if os.environ.get("NB_TEST") == "1":      # only used when testing outside Jupyter
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:                        # plain Python fallback
    display = print

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": 0.3})
TRADING_DAYS = 252
WARNINGS = []


def check(cond, msg):
    if not cond:
        raise ValueError(msg)


def warn(msg):
    WARNINGS.append(msg)
    print("WARNING:", msg)


def to_date(x):
    if x is None:
        return None
    if isinstance(x, dt.datetime):
        return x.date()
    if isinstance(x, dt.date):
        return x
    if isinstance(x, (int, float)):
        raise ValueError(
            f"A date must be written inside quotes, like \"2026-10-27\". "
            f"Without quotes Python does the subtraction 2026-10-27 and gets {x}, which is not a date.")
    text = str(x).strip()
    for pattern in ("%Y-%m-%d", "%d-%m-%Y", "%d/%m/%Y", "%d-%b-%Y"):    # 2026-10-27, 27-10-2026, 27/10/2026, 27-Oct-2026
        try:
            return dt.datetime.strptime(text, pattern).date()
        except ValueError:
            pass
    raise ValueError(f"Cannot read the date {text!r}. Use \"YYYY-MM-DD\", for example \"2026-10-27\".")


def last_tuesday(year, month):
    first_next = dt.date(year + (month == 12), month % 12 + 1, 1)
    d = first_next - dt.timedelta(days=1)
    while d.weekday() != 1:                # Tuesday
        d -= dt.timedelta(days=1)
    return d


def monthly_expiry(year, month, holidays=None):
    # NSE stock derivatives expire on the last Tuesday of the month.
    # If that day is a weekend/holiday, the expiry moves to the previous trading day.
    holidays = {to_date(h) for h in (EXCHANGE_HOLIDAYS if holidays is None else holidays)}
    d = last_tuesday(year, month)
    while d.weekday() >= 5 or d in holidays:
        d -= dt.timedelta(days=1)
    return d


def next_monthly_expiry(valuation_date, min_days=1, holidays=None):
    y, m = valuation_date.year, valuation_date.month
    for _ in range(24):
        e = monthly_expiry(y, m, holidays)
        if (e - valuation_date).days >= min_days:
            return e
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    raise RuntimeError("Could not find an expiry date")


def fmt(x, nd=2):
    return "n/a" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:,.{nd}f}"


# Validate the config early so mistakes show up with a clear message
check(OPTION_TYPE in ("call", "put"), "OPTION_TYPE must be 'call' or 'put'")
check(EXERCISE in ("european", "american"), "EXERCISE must be 'european' or 'american'")
check(DATA_MODE in ("live", "manual", "auto"), "DATA_MODE must be 'live', 'manual' or 'auto'")
check(VOL_METHOD in ("close", "ewma", "parkinson", "implied"), "VOL_METHOD must be 'close', 'ewma', 'parkinson' or 'implied'")
check(STRIKE > 0 and LOT_SIZE > 0 and TREE_STEPS >= 1 and VOL_WINDOW >= 5, "STRIKE, LOT_SIZE, TREE_STEPS, VOL_WINDOW must be positive")
print("Config OK")

## 3. Volatility estimators
Volatility = how much the stock typically moves in a year (as a decimal, 0.21 = 21%).
It is the most important input, and the only one that is really an estimate.
These functions measure it from PAST prices (historical volatility). Instead, `VOL_METHOD = "implied"`
takes it from a real option's market price (implied volatility). See IV_REFERENCE in the CONFIG cell.

In [ ]:
def log_returns(hist):
    c = hist["Close"].astype(float)
    r = np.log(c / c.shift(1))
    if "gap" in hist.columns:
        r = r[~hist["gap"].astype(bool)]    # skip returns that span days missing from the data
    return r.dropna()


def vol_close(hist, window):
    r = log_returns(hist).iloc[-window:]
    return float(r.std(ddof=1) * math.sqrt(TRADING_DAYS)), len(r)


def vol_ewma(hist, window=None, lam=0.94):
    r = log_returns(hist).iloc[-250:]
    w = lam ** np.arange(len(r))[::-1]
    var = float(np.sum(w * r.values ** 2) / np.sum(w))
    return math.sqrt(var * TRADING_DAYS), len(r)


def vol_parkinson(hist, window):
    if not {"High", "Low"}.issubset(hist.columns):
        return float("nan"), 0
    h = hist[["High", "Low"]].iloc[-window:]
    x = np.log(h["High"] / h["Low"]) ** 2
    return math.sqrt(float(x.mean()) / (4 * math.log(2)) * TRADING_DAYS), len(x)


def estimate_vol(hist, method, window):
    fn = {"close": vol_close, "ewma": vol_ewma, "parkinson": vol_parkinson}[method]
    v, n = fn(hist, window)
    if math.isnan(v):
        warn(f"VOL_METHOD='{method}' needs data your history does not have. Using 'close' instead.")
        v, n = vol_close(hist, window)
    if n < min(window, 30):
        warn(f"Only {n} daily returns available for the volatility estimate. It may be unreliable.")
    return v, n


def vol_table(hist):
    rows = []
    for w in (21, 63, 126):
        if len(hist) > w:
            rows.append((f"Close-to-close, last {w} days", fmt(vol_close(hist, w)[0] * 100) + "%"))
    rows.append(("Close-to-close, all history", fmt(vol_close(hist, len(hist))[0] * 100) + "%"))
    rows.append(("EWMA (lambda 0.94)", fmt(vol_ewma(hist)[0] * 100) + "%"))
    p = vol_parkinson(hist, VOL_WINDOW)[0]
    rows.append((f"Parkinson, last {VOL_WINDOW} days", "needs High/Low" if math.isnan(p) else fmt(p * 100) + "%"))
    return pd.DataFrame(rows, columns=["Estimate", "Annual volatility"])

## 4. Pricing engine
* `binomial_price` is the tree from the lesson, written fast (it works on whole rows at once).
* `u` and `d` come from the volatility: `u = e^(sigma*sqrt(dt))`, `d = 1/u`.
* `bs_price` is the Black-Scholes formula, used to double-check European prices.
* `implied_vol` runs the model backwards: market price in, volatility out.

In [ ]:
def _norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def _norm_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)


def _intrinsic(S, K, option):
    return np.maximum(S - K, 0.0) if option == "call" else np.maximum(K - S, 0.0)


def _tree_params(T, r, q, sigma, N, u=None, d=None):
    dt_ = T / N
    if u is None or d is None:
        u = math.exp(sigma * math.sqrt(dt_))
        d = 1.0 / u
    p = (math.exp((r - q) * dt_) - d) / (u - d)
    if not (0.0 < p < 1.0):
        raise ValueError(f"Risk-neutral probability p = {p:.4f} is outside 0..1. "
                         "Use more TREE_STEPS (or check the rate and volatility).")
    return dt_, u, d, p


def binomial_price(S, K, T, r, q, sigma, N=500, option="call", exercise="european", u=None, d=None):
    dt_, u, d, p = _tree_params(T, r, q, sigma, N, u, d)
    disc = math.exp(-r * dt_)
    j = np.arange(N + 1)
    V = _intrinsic(S * np.power(u, j) * np.power(d, N - j), K, option)   # value at expiry for j up-moves
    american = exercise == "american"
    for step in range(N - 1, -1, -1):
        V = disc * (p * V[1:step + 2] + (1.0 - p) * V[:step + 1])
        if american:
            i = np.arange(step + 1)
            Sj = S * np.power(u, i) * np.power(d, step - i)
            V = np.maximum(V, _intrinsic(Sj, K, option))
    return float(V[0])


def tree_tables(S, K, T, r, q, sigma, N, option, exercise):
    dt_, u, d, p = _tree_params(T, r, q, sigma, N)
    disc = math.exp(-r * dt_)
    stock = np.full((N + 1, N + 1), np.nan)
    val = np.full((N + 1, N + 1), np.nan)
    for j in range(N + 1):
        for i in range(j + 1):
            stock[j, i] = S * u ** i * d ** (j - i)
    for i in range(N + 1):
        val[N, i] = float(_intrinsic(stock[N, i], K, option))
    for j in range(N - 1, -1, -1):
        for i in range(j + 1):
            hold = disc * (p * val[j + 1, i + 1] + (1 - p) * val[j + 1, i])
            val[j, i] = max(hold, float(_intrinsic(stock[j, i], K, option))) if exercise == "american" else hold
    return stock, val, dict(dt=dt_, u=u, d=d, p=p)


def bs_price(S, K, T, r, q, sigma, option="call"):
    if T <= 0 or sigma <= 0:
        fwd = S * math.exp(-q * T) - K * math.exp(-r * T)
        return max(fwd, 0.0) if option == "call" else max(-fwd, 0.0)
    sq = sigma * math.sqrt(T)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / sq
    d2 = d1 - sq
    if option == "call":
        return S * math.exp(-q * T) * _norm_cdf(d1) - K * math.exp(-r * T) * _norm_cdf(d2)
    return K * math.exp(-r * T) * _norm_cdf(-d2) - S * math.exp(-q * T) * _norm_cdf(-d1)


def fast_price(S, K, T, r, q, sigma, option, exercise, N=200):
    # Black-Scholes is exact and instant for European options, so use it when possible
    if exercise == "european":
        return bs_price(S, K, T, r, q, sigma, option)
    return binomial_price(S, K, T, r, q, sigma, N, option, exercise)


def greeks(S, K, T, r, q, sigma, option, exercise, N=400):
    # delta: change per Rs 1 move of the stock.  gamma: change of delta per Rs 1.
    # vega: per 1 volatility point.  theta: per calendar day.  rho: per 1% change in the rate.
    if exercise == "european":
        sq = sigma * math.sqrt(T)
        d1 = (math.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / sq
        d2 = d1 - sq
        eq, er = math.exp(-q * T), math.exp(-r * T)
        pdf = _norm_pdf(d1)
        if option == "call":
            delta = eq * _norm_cdf(d1)
            theta = -S * eq * pdf * sigma / (2 * math.sqrt(T)) - r * K * er * _norm_cdf(d2) + q * S * eq * _norm_cdf(d1)
            rho = K * T * er * _norm_cdf(d2)
        else:
            delta = eq * (_norm_cdf(d1) - 1.0)
            theta = -S * eq * pdf * sigma / (2 * math.sqrt(T)) + r * K * er * _norm_cdf(-d2) - q * S * eq * _norm_cdf(-d1)
            rho = -K * T * er * _norm_cdf(-d2)
        return dict(delta=delta, gamma=eq * pdf / (S * sq), vega=S * eq * pdf * math.sqrt(T) / 100.0,
                    theta=theta / 365.0, rho=rho / 100.0)
    f = lambda S_=S, T_=T, r_=r, s_=sigma: binomial_price(S_, K, T_, r_, q, s_, N, option, exercise)
    h = 0.01 * S
    up, mid, dn = f(S_=S + h), f(), f(S_=S - h)
    return dict(delta=(up - dn) / (2 * h), gamma=(up - 2 * mid + dn) / h ** 2,
                vega=(f(s_=sigma + 0.01) - f(s_=max(sigma - 0.01, 1e-4))) / 2.0,
                theta=f(T_=max(T - 1 / 365, 1e-6)) - mid, rho=(f(r_=r + 0.01) - f(r_=r - 0.01)) / 2.0)


def implied_vol(price, S, K, T, r, q, option, exercise, N=300, lo=1e-4, hi=5.0):
    # Runs the model backwards: finds the volatility that makes the model price equal the market price
    if exercise == "european":
        floor = max(S * math.exp(-q * T) - K * math.exp(-r * T), 0.0) if option == "call" else max(K * math.exp(-r * T) - S * math.exp(-q * T), 0.0)
        ceil = S * math.exp(-q * T) if option == "call" else K * math.exp(-r * T)
    else:
        floor = max(S - K, 0.0) if option == "call" else max(K - S, 0.0)
        ceil = S if option == "call" else K
    if price < floor - 1e-9:
        return float("nan"), f"market price {price:.2f} is below the minimum possible value {floor:.2f} (check inputs or market price)"
    if price > ceil + 1e-9:
        return float("nan"), f"market price {price:.2f} is above the maximum possible value {ceil:.2f}"
    if exercise == "american":
        # The tree needs sigma * sqrt(dt) to be larger than the interest drift per step, or p leaves 0..1.
        # With very short expiries that rules out extremely tiny volatilities, so start the search higher.
        lo = max(lo, 2.0 * abs(r - q) * math.sqrt(T / N))
    g = lambda s: fast_price(S, K, T, r, q, s, option, exercise, N) - price
    if g(lo) > 0:
        return float("nan"), "price is at or below the zero-volatility value; no implied volatility"
    if g(hi) < 0:
        return float("nan"), "price needs more than 500% volatility"
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        gm = g(mid)
        if abs(gm) < 1e-9 or (hi - lo) < 1e-12:
            break
        if gm > 0:
            hi = mid
        else:
            lo = mid
    return mid, "ok"


def reference_vol(ref, S, T, r, q, exercise):
    # Volatility from a real option's market price (used by VOL_METHOD = "implied")
    option, K = ref["option"], float(ref["strike"])
    check(option in ("call", "put"), "IV_REFERENCE['option'] must be 'call' or 'put'")
    bid, ask, last = ref.get("bid"), ref.get("ask"), ref.get("price")
    if bid and ask and bid > 0 and ask >= bid:
        price = (bid + ask) / 2.0
    elif last and last > 0:
        price = float(last)
    else:
        raise ValueError("VOL_METHOD is 'implied' but IV_REFERENCE has no market price. "
                         "Fill in 'price' (or both 'bid' and 'ask') from your broker's option chain, "
                         "or set VOL_METHOD back to 'close'.")
    iv, msg = implied_vol(price, S, K, T, r, q, option, exercise)
    check(not math.isnan(iv), f"Cannot get an implied volatility from IV_REFERENCE: {msg}")
    return iv, price

## 5. Get the inputs (live or manual)

In [ ]:
def _clean_history(df):
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    if getattr(df.index, "tz", None) is not None:
        df.index = df.index.tz_localize(None)
    df.index = df.index.normalize()
    df = df.sort_index()
    df = df[~df.index.duplicated(keep="last")]
    df = df[df["Close"].notna() & (df["Close"] > 0)]
    if "Volume" in df.columns:
        df = df[df["Volume"].fillna(0) > 0]   # Yahoo shows flat zero-volume rows on exchange holidays
    df["gap"] = False
    return df


def fetch_live_history(ticker, period="2y"):
    import yfinance as yf
    df = yf.Ticker(ticker).history(period=period, interval="1d", auto_adjust=True)
    if df is None or df.empty:
        raise RuntimeError("Yahoo Finance returned no data (wrong ticker, or no internet?)")
    return _clean_history(df)


def fetch_live_spot(ticker):
    import yfinance as yf
    t = yf.Ticker(ticker)
    try:
        fi = t.fast_info
        for key in ("last_price", "lastPrice"):
            try:
                p = float(fi[key])
                if p > 0:
                    return p, "live (Yahoo last price)"
            except Exception:
                pass
    except Exception:
        pass
    raise RuntimeError("could not read a live last price")


def load_history_csv(path):
    raw = pd.read_csv(path)
    low = {c.lower().strip(): c for c in raw.columns}
    check("date" in low, f"{path}: needs a 'Date' column")
    close_col = low.get("adj close") or low.get("close")
    check(close_col is not None, f"{path}: needs a 'Close' column")
    out = pd.DataFrame({"Close": raw[close_col].astype(float).values}, index=pd.to_datetime(raw[low["date"]]))
    for name in ("open", "high", "low", "volume"):
        if name in low:
            out[name.title()] = raw[low[name]].values
    out["gap"] = raw[low["gap"]].fillna(0).astype(int).astype(bool).values if "gap" in low else False
    out.index.name = "Date"
    return out.sort_index()


def fetch_nse_option_quote(symbol, expiry_date, strike, option):
    # EXPERIMENTAL. NSE changes its website often and may block scripts. Check NSE's terms of use.
    # For serious use, take prices from your broker's API instead.
    import requests
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.nseindia.com/option-chain",
    })
    s.get("https://www.nseindia.com/option-chain", timeout=15)       # collects cookies
    resp = s.get(f"https://www.nseindia.com/api/option-chain-equities?symbol={symbol}", timeout=15)
    resp.raise_for_status()
    return parse_nse_chain(resp.json()["records"]["data"], symbol, expiry_date, strike, option)


def parse_nse_chain(rows, symbol, expiry_date, strike, option):
    want = expiry_date.strftime("%d-%b-%Y")
    side = "CE" if option == "call" else "PE"
    for row in rows:
        if row.get("expiryDate") == want and float(row.get("strikePrice", -1)) == float(strike) and side in row:
            q = row[side]
            return {"last": q.get("lastPrice"), "bid": q.get("bidprice", q.get("bidPrice")),
                    "ask": q.get("askPrice"), "iv": q.get("impliedVolatility")}
    raise LookupError(f"{symbol} {strike} {side} expiring {want} not found in NSE's option chain")


def resolve_inputs(valuation_date):
    hist, hist_src, live_ok = None, "none", False

    if DATA_MODE in ("live", "auto"):
        try:
            hist = fetch_live_history(YF_TICKER)
            hist_src, live_ok = f"live (Yahoo Finance, {YF_TICKER})", True
        except Exception as e:
            if DATA_MODE == "live":
                raise RuntimeError(f"DATA_MODE='live' but the download failed: {e}\n"
                                   "Fix: install yfinance, check internet, or set DATA_MODE='manual' and fill MANUAL.") from e
            warn(f"Live download failed ({type(e).__name__}: {e}). Falling back to your MANUAL inputs.")

    if hist is None:
        path = MANUAL.get("price_csv")
        if path and os.path.exists(path):
            hist = load_history_csv(path)
            hist_src = f"CSV file ({path})"
        elif path:
            warn(f"price_csv '{path}' not found. Continuing without price history.")

    if hist is not None and VALUATION_DATE is not None:
        hist = hist[hist.index.date <= valuation_date]     # pricing as of a past day: ignore later data

    # --- spot
    if live_ok and VALUATION_DATE is None:
        try:
            spot, spot_src = fetch_live_spot(YF_TICKER)
        except Exception:
            spot, spot_src = float(hist["Close"].iloc[-1]), "last close in live history"
    elif MANUAL.get("spot") is not None:
        spot, spot_src = float(MANUAL["spot"]), "manual"
    elif hist is not None and len(hist):
        spot, spot_src = float(hist["Close"].iloc[-1]), "last close in history"
    else:
        raise ValueError("No stock price available. Set MANUAL['spot'] or provide live data / a price CSV.")

    # --- history freshness
    if hist is not None and len(hist):
        age = (valuation_date - hist.index[-1].date()).days
        if age > 5:
            warn(f"Price history ends {age} days before the valuation date. Update it for an accurate volatility.")

    # --- volatility
    if MANUAL.get("volatility") is not None:
        vol, vol_src = float(MANUAL["volatility"]), "manual"
        if VOL_METHOD == "implied":
            warn("MANUAL['volatility'] is set, so it is used instead of VOL_METHOD='implied'. Set it to None to use the implied volatility.")
    elif VOL_METHOD == "implied":
        vol, vol_src = None, "implied, worked out below"
    else:
        check(hist is not None and len(hist) > 10,
              "No price history to estimate volatility. Fix one of: (1) get live data (DATA_MODE='live' or 'auto', internet, yfinance), "
              "(2) point MANUAL['price_csv'] to a valid CSV (run the notebook from the folder that contains data/), "
              "or (3) type your own number in MANUAL['volatility'].")
        vol, n = estimate_vol(hist, VOL_METHOD, VOL_WINDOW)
        vol_src = f"estimated: {VOL_METHOD}, {n} daily returns"

    # --- rate and dividend yield
    check(MANUAL.get("risk_free_rate") is not None, "Set MANUAL['risk_free_rate'] (decimal, e.g. 0.0557)")
    r = float(MANUAL["risk_free_rate"])
    q = float(MANUAL.get("dividend_yield") or 0.0)

    # --- market price of the option
    last, bid, ask = MANUAL.get("market_option_price"), MANUAL.get("market_bid"), MANUAL.get("market_ask")
    if FETCH_NSE_QUOTE and last is None and bid is None and ask is None:
        try:
            qte = fetch_nse_option_quote(SYMBOL, EXPIRY, STRIKE, OPTION_TYPE)
            last, bid, ask = qte["last"], qte["bid"], qte["ask"]
            print(f"NSE quote: last {last}, bid {bid}, ask {ask}, NSE implied vol {qte['iv']}")
        except Exception as e:
            warn(f"NSE quote fetch failed ({type(e).__name__}: {e}). Type the price into MANUAL instead.")
    market_price, market_kind, market_spread = None, None, None
    if bid and ask and bid > 0 and ask >= bid:
        market_price, market_kind, market_spread = (bid + ask) / 2, "middle of bid and ask", ask - bid
    elif last and last > 0:
        market_price, market_kind = float(last), "last traded price"
        if bid is not None or ask is not None:
            warn("Bid/ask looks incomplete, so the last traded price is used.")

    # --- sanity checks that catch the most common typing mistakes
    check(spot > 0, "Stock price must be positive")
    check(vol is None or 0 < vol < 3, f"Volatility {vol} looks wrong. Use a decimal: 0.21 means 21%.")
    check(-0.05 < r < 0.5, f"Risk-free rate {r} looks wrong. Use a decimal: 0.0557 means 5.57%.")
    check(0 <= q < 0.5, f"Dividend yield {q} looks wrong. Use a decimal: 0.01 means 1%.")

    return dict(hist=hist, hist_src=hist_src, spot=spot, spot_src=spot_src, vol=vol, vol_src=vol_src,
                r=r, q=q, market_price=market_price, market_kind=market_kind, market_spread=market_spread)


VAL_DATE = to_date(VALUATION_DATE) or dt.date.today()
EXPIRY = to_date(EXPIRY_DATE) or next_monthly_expiry(VAL_DATE)
check(EXPIRY > VAL_DATE, f"Expiry {EXPIRY} must be after the valuation date {VAL_DATE}")
DAYS = (EXPIRY - VAL_DATE).days
T = DAYS / 365.0

INP = resolve_inputs(VAL_DATE)
HIST, S0, SIGMA, R, Q = INP["hist"], INP["spot"], INP["vol"], INP["r"], INP["q"]
MARKET_PRICE, MARKET_KIND, MARKET_SPREAD = INP["market_price"], INP["market_kind"], INP["market_spread"]

# Known cash dividends before expiry: remove their present value from the share price
DIV_LIST = [(to_date(d), float(a)) for d, a in DIVIDENDS if VAL_DATE < to_date(d) <= EXPIRY]
PV_DIV = sum(a * math.exp(-R * (d - VAL_DATE).days / 365.0) for d, a in DIV_LIST)
S_MODEL = S0 - PV_DIV
check(S_MODEL > 0, "Dividends are larger than the share price. Check DIVIDENDS.")
HIST_VOL = None
if SIGMA is None:                                   # VOL_METHOD = "implied"
    SIGMA, _ref_price = reference_vol(IV_REFERENCE, S_MODEL, T, R, Q, EXERCISE)
    check(0 < SIGMA < 3, f"Implied volatility {SIGMA} looks wrong. Check the prices in IV_REFERENCE.")
    INP["vol_src"] = f"implied from the {float(IV_REFERENCE['strike']):,.0f} {IV_REFERENCE['option']}, market price {_ref_price:,.2f}"
    if HIST is not None and len(HIST) > 10:
        HIST_VOL = vol_close(HIST, VOL_WINDOW)[0]   # shown only for comparison
    if IV_REFERENCE["option"] == OPTION_TYPE and float(IV_REFERENCE["strike"]) == float(STRIKE):
        warn("IV_REFERENCE is the same option you are pricing, so the model price will match its market price by construction "
             "and the verdict will always be 'about fair'. Use a different option's implied volatility (for example the at-the-money one).")
if DAYS < 7:
    warn(f"Only {DAYS} days to expiry. Prices and Greeks move very fast this close to expiry.")
if EXERCISE == "american" and DIV_LIST:
    warn("American options with discrete dividends are only approximated here.")

summary = pd.DataFrame([
    ("Option",           f"{EXERCISE.title()} {OPTION_TYPE}, strike {STRIKE:,.0f}"),
    ("Valuation date",   f"{VAL_DATE}"),
    ("Expiry",           f"{EXPIRY:%a %d %b %Y} ({DAYS} days, T = {T:.4f} years)"
                         + ("  [picked automatically, confirm with your broker]" if to_date(EXPIRY_DATE) is None else "")),
    ("Stock price",      f"{S0:,.2f}   [{INP['spot_src']}]"),
    ("Volatility",       f"{SIGMA * 100:.2f}%   [{INP['vol_src']}]"),
    *([("Historical volatility (for comparison)", f"{HIST_VOL * 100:.2f}%   [close-to-close, last {VOL_WINDOW} days]")] if HIST_VOL else []),
    ("Risk-free rate",   f"{R * 100:.2f}%   [manual]"),
    ("Dividend yield",   f"{Q * 100:.2f}%   [manual]"),
    ("Known dividends",  f"{len(DIV_LIST)} before expiry, present value {PV_DIV:.2f}" if DIV_LIST else "none"),
    ("Price history",    INP["hist_src"] + (f", {len(HIST)} days, last {HIST.index[-1].date()}" if HIST is not None else "")),
    ("Market price",     f"{MARKET_PRICE:,.2f}   [{MARKET_KIND}]" if MARKET_PRICE else "not entered"),
], columns=["Input", "Value  [source]"])
display(summary)
if HIST is not None:
    display(vol_table(HIST))

## 6. Self-tests
These check the engine against known answers every time you run the notebook.
If one fails, do not trust the results below it.

In [ ]:
def run_self_tests():
    results = []

    def t(name, ok, detail):
        results.append((name, "PASS" if ok else "FAIL", detail))

    lesson = binomial_price(100, 90, 10, 0.0, 0.0, 0.2, N=10, option="call", u=1.2, d=0.8)
    t("Course example (u=1.2, d=0.8) gives 29.38", abs(lesson - 29.3845445) < 1e-4, f"{lesson:.4f}")

    S, K_, T_, r_, q_, s_ = 100.0, 100.0, 1.0, 0.05, 0.02, 0.25
    for opt in ("call", "put"):
        b, a = binomial_price(S, K_, T_, r_, q_, s_, 1000, opt), bs_price(S, K_, T_, r_, q_, s_, opt)
        t(f"Tree (N=1000) matches Black-Scholes, {opt}", abs(b - a) < 0.02, f"tree {b:.4f} vs BS {a:.4f}")

    c, p = binomial_price(S, K_, T_, r_, q_, s_, 400, "call"), binomial_price(S, K_, T_, r_, q_, s_, 400, "put")
    parity = S * math.exp(-q_ * T_) - K_ * math.exp(-r_ * T_)
    t("Put-call parity holds", abs((c - p) - parity) < 1e-8, f"C-P = {c - p:.6f}, expected {parity:.6f}")

    for opt in ("call", "put"):
        iv, msg = implied_vol(fast_price(S, K_, T_, r_, q_, 0.31, opt, "european"), S, K_, T_, r_, q_, opt, "european")
        t(f"Implied vol round trip, {opt}", abs(iv - 0.31) < 1e-6, f"recovered {iv:.6f}")

    ap = binomial_price(S, K_, T_, r_, 0.0, s_, 400, "put", "american")
    ep = binomial_price(S, K_, T_, r_, 0.0, s_, 400, "put", "european")
    t("American put >= European put", ap >= ep - 1e-12, f"{ap:.4f} >= {ep:.4f}")
    ac = binomial_price(S, K_, T_, r_, 0.0, s_, 400, "call", "american")
    ec = binomial_price(S, K_, T_, r_, 0.0, s_, 400, "call", "european")
    t("American call = European call when no dividends", abs(ac - ec) < 1e-8, f"{ac:.4f} vs {ec:.4f}")

    ga = greeks(S, K_, T_, r_, q_, s_, "call", "european")["delta"]
    gn = greeks(S, K_, T_, r_, q_, s_, "call", "american", 400)["delta"]
    t("Delta formula agrees with tree (no early exercise gain)", abs(ga - gn) < 0.01, f"{ga:.4f} vs {gn:.4f}")

    t("Expiry helper: last Tuesday of Dec 2026 is 29 Dec", last_tuesday(2026, 12) == dt.date(2026, 12, 29), str(last_tuesday(2026, 12)))
    t("Expiry helper: next monthly expiry from 21 Oct 2026 is 27 Oct", next_monthly_expiry(dt.date(2026, 10, 21)) == dt.date(2026, 10, 27),
      str(next_monthly_expiry(dt.date(2026, 10, 21))))
    t("Expiry helper: after 27 Oct the next expiry is 23 Nov (24 Nov is a holiday)", next_monthly_expiry(dt.date(2026, 10, 27)) == dt.date(2026, 11, 23),
      str(next_monthly_expiry(dt.date(2026, 10, 27))))
    t("Expiry helper: with no holidays listed, the last Tuesday is used", next_monthly_expiry(dt.date(2026, 10, 27), holidays=[]) == dt.date(2026, 11, 24),
      str(next_monthly_expiry(dt.date(2026, 10, 27), holidays=[])))

    same = {to_date(s) for s in ("2026-10-27", "27-10-2026", "27/10/2026", "27-Oct-2026")}
    t("Date reader accepts 4 common formats", same == {dt.date(2026, 10, 27)}, str(sorted(same)))
    try:
        to_date(2026 - 10 - 27)                       # what Python makes of an unquoted date
        caught = False
    except ValueError:
        caught = True
    t("Date reader rejects an unquoted date (2026-10-27 = 1989)", caught, "raises a clear error")

    short_p = binomial_price(1200, 1200, 6 / 365, 0.0557, 0.0, 0.25, N=300, option="put", exercise="american")
    short_iv, _ = implied_vol(short_p, 1200, 1200, 6 / 365, 0.0557, 0.0, "put", "american")
    t("Implied vol works for a 6-day American put", abs(short_iv - 0.25) < 1e-4, f"recovered {short_iv:.6f}")

    ref_px = bs_price(1200, 1250, 0.25, 0.0557, 0.0, 0.27, "call")
    ref_iv, _ = reference_vol({"option": "call", "strike": 1250, "price": None, "bid": ref_px - 0.5, "ask": ref_px + 0.5},
                              1200, 0.25, 0.0557, 0.0, "european")
    t("Volatility from a reference option (bid/ask middle) is recovered", abs(ref_iv - 0.27) < 1e-6, f"recovered {ref_iv:.6f}")
    try:
        reference_vol({"option": "call", "strike": 1250, "price": None, "bid": None, "ask": None}, 1200, 0.25, 0.0557, 0.0, "european")
        no_price_caught = False
    except ValueError:
        no_price_caught = True
    t("Missing reference price gives a clear error", no_price_caught, "raises a clear error")

    out = pd.DataFrame(results, columns=["Test", "Result", "Detail"])
    display(out)
    if (out["Result"] == "FAIL").any():
        raise AssertionError("A self-test failed. Do not use the results until this is fixed.")
    print("All self-tests passed.")


run_self_tests()

## 7. Price your option

In [ ]:
model_price = binomial_price(S_MODEL, STRIKE, T, R, Q, SIGMA, TREE_STEPS, OPTION_TYPE, EXERCISE)
bs_ref = bs_price(S_MODEL, STRIKE, T, R, Q, SIGMA, OPTION_TYPE) if EXERCISE == "european" else None
intrinsic_now = float(_intrinsic(S0, STRIKE, OPTION_TYPE))

rows = [
    ("Model price per share (tree)", f"Rs {model_price:,.2f}"),
    ("Value of one lot",             f"Rs {model_price * LOT_SIZE:,.0f}  ({LOT_SIZE} shares)"),
    ("Intrinsic value (worth if exercised now)", f"Rs {intrinsic_now:,.2f}"),
    ("Time value",                   f"Rs {model_price - intrinsic_now:,.2f}"),
    ("Moneyness (stock / strike)",   f"{S0 / STRIKE:.3f}"),
]
if bs_ref is not None:
    rows.append(("Black-Scholes check", f"Rs {bs_ref:,.2f}  (difference {model_price - bs_ref:+.3f})"))
    if abs(model_price - bs_ref) > 0.02 * max(bs_ref, 1.0):
        warn("Tree and Black-Scholes differ by more than 2%. Increase TREE_STEPS.")
display(pd.DataFrame(rows, columns=["Result", "Value"]))

print("\nA small 3-step tree so you can see every node (stock price, then option value):")
stock3, val3, par3 = tree_tables(S_MODEL, STRIKE, T, R, Q, SIGMA, 3, OPTION_TYPE, EXERCISE)
print(f"   dt = {par3['dt']:.4f} yr   u = {par3['u']:.5f}   d = {par3['d']:.5f}   p (risk-neutral) = {par3['p']:.4f}")
cols = [f"{i} up" for i in range(4)]
idx = [f"step {j}" for j in range(4)]
display(pd.DataFrame(stock3, index=idx, columns=cols).round(2).fillna(""))
display(pd.DataFrame(val3, index=idx, columns=cols).round(2).fillna(""))
print("(The 3-step price is rough. The 500-step price above is the one to use.)")

## 8. Market price vs model price
"Cheap" or "expensive" here only means *compared with the volatility you chose*.

In [ ]:
IV = float("nan")
if MARKET_PRICE is None:
    print("No market price entered.")
    print("Type the option's price into MANUAL['market_option_price'] (or market_bid and market_ask) and run again.")
else:
    IV, msg = implied_vol(MARKET_PRICE, S_MODEL, STRIKE, T, R, Q, OPTION_TYPE, EXERCISE)
    edge = model_price - MARKET_PRICE
    buffer = max(EDGE_BUFFER_PCT / 100.0 * model_price, (MARKET_SPREAD or 0.0) / 2.0)
    verdict = ("CHEAP compared with the volatility used above" if edge > buffer else
               "EXPENSIVE compared with the volatility used above" if edge < -buffer else
               "ABOUT FAIR compared with the volatility used above")
    print(f"Market price        : Rs {MARKET_PRICE:,.2f}  ({MARKET_KIND})")
    print(f"Model price         : Rs {model_price:,.2f}  (volatility {SIGMA * 100:.2f}%, {INP['vol_src']})")
    print(f"Model - market      : Rs {edge:+,.2f} per share, Rs {edge * LOT_SIZE:+,.0f} per lot")
    print(f"Ignore gaps below   : Rs {buffer:,.2f}  (your buffer, or half the bid-ask gap)")
    if math.isnan(IV):
        print(f"Implied volatility  : not available ({msg})")
    else:
        print(f"Implied volatility  : {IV * 100:.2f}%  (what the market is assuming)  vs the {SIGMA * 100:.2f}% used above")
    print(f"\nVerdict: {verdict}")
    print("Remember: if the market knows something your history does not (results, news), the gap is not an error.")

## 9. Greeks (how the price reacts)

In [ ]:
g_hist = greeks(S_MODEL, STRIKE, T, R, Q, SIGMA, OPTION_TYPE, EXERCISE)
cols_out = {"Greek": ["Delta", "Gamma", "Vega (per 1 vol point)", "Theta (per day)", "Rho (per 1% rate)"],
            f"At your volatility ({SIGMA * 100:.1f}%)": [g_hist[k] for k in ("delta", "gamma", "vega", "theta", "rho")]}
if not math.isnan(IV):
    g_iv = greeks(S_MODEL, STRIKE, T, R, Q, IV, OPTION_TYPE, EXERCISE)
    cols_out[f"At implied volatility ({IV * 100:.1f}%)"] = [g_iv[k] for k in ("delta", "gamma", "vega", "theta", "rho")]
gdf = pd.DataFrame(cols_out)
for c in gdf.columns[1:]:
    gdf[c] = gdf[c].map(lambda v: f"{v:,.4f}")
display(gdf)
print(f"Delta {g_hist['delta']:.2f}: if the stock moves Rs 10, the option moves about Rs {g_hist['delta'] * 10:.2f}.")
print(f"To hedge ONE short option, hold about {abs(g_hist['delta']):.2f} shares ({abs(g_hist['delta']) * LOT_SIZE:,.0f} per lot).")
print(f"Theta {g_hist['theta']:.2f}: the option loses about Rs {abs(g_hist['theta']):.2f} per day from time passing, all else equal.")

## 10. Charts

In [ ]:
if HIST is not None and len(HIST) > 30:
    fig, ax1 = plt.subplots()
    ax1.plot(HIST.index, HIST["Close"], color="tab:blue")
    ax1.set_ylabel("Close price (Rs)", color="tab:blue")
    ax2 = ax1.twinx()
    roll = log_returns(HIST).rolling(21).std() * math.sqrt(TRADING_DAYS) * 100
    ax2.plot(roll.index, roll.values, color="tab:orange", alpha=0.8)
    ax2.axhline(SIGMA * 100, color="tab:orange", ls="--", lw=1)
    ax2.set_ylabel("21-day volatility (%)  dashed = value used", color="tab:orange")
    ax2.grid(False)
    ax1.set_title(f"{SYMBOL}: price and rolling volatility")
    plt.tight_layout()
    plt.show()

# convergence
Ns = [5, 10, 25, 50, 100, 200, 400, 800]
conv = [binomial_price(S_MODEL, STRIKE, T, R, Q, SIGMA, n, OPTION_TYPE, EXERCISE) for n in Ns]
plt.figure()
plt.plot(Ns, conv, "o-", label="binomial tree")
if bs_ref is not None:
    plt.axhline(bs_ref, color="red", ls="--", label="Black-Scholes")
plt.xscale("log")
plt.xlabel("number of steps (N)")
plt.ylabel("option price (Rs)")
plt.title("The tree price settles as the tree gets finer")
plt.legend()
plt.tight_layout()
plt.show()

# price vs stock price, today and at expiry
grid = np.linspace(0.75 * S0, 1.25 * S0, 120)
today_curve = [fast_price(s - PV_DIV, STRIKE, T, R, Q, SIGMA, OPTION_TYPE, EXERCISE) for s in grid]
plt.figure()
plt.plot(grid, today_curve, label="value today (model)")
plt.plot(grid, _intrinsic(grid, STRIKE, OPTION_TYPE), "--", label="value at expiry")
plt.axvline(S0, color="grey", lw=1)
plt.scatter([S0], [model_price], color="black", zorder=5, label="now")
if MARKET_PRICE is not None:
    plt.scatter([S0], [MARKET_PRICE], color="red", zorder=5, label="market price")
plt.xlabel("stock price (Rs)")
plt.ylabel("option value per share (Rs)")
plt.title(f"{OPTION_TYPE.title()} option, strike {STRIKE:,.0f}")
plt.legend()
plt.tight_layout()
plt.show()

# price vs volatility
vgrid = np.linspace(0.05, 0.60, 80)
vcurve = [fast_price(S_MODEL, STRIKE, T, R, Q, v, OPTION_TYPE, EXERCISE) for v in vgrid]
plt.figure()
plt.plot(vgrid * 100, vcurve)
plt.axvline(SIGMA * 100, color="tab:orange", ls="--", label=f"your volatility {SIGMA * 100:.1f}%")
if not math.isnan(IV):
    plt.axvline(IV * 100, color="red", ls="--", label=f"implied volatility {IV * 100:.1f}%")
plt.xlabel("volatility (% per year)")
plt.ylabel("option price (Rs)")
plt.title("The volatility you choose drives the price")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Strategy builder
Pick a strategy in the CONFIG cell (`STRATEGY`). The chart shows profit or loss per lot.
Premiums come from the model (flat volatility), except the leg that matches your option when a market price is given.

In [ ]:
def build_legs(name):
    K, w = STRIKE, STRATEGY_WIDTH
    presets = {
        "covered_call":     [dict(kind="stock", qty=1), dict(kind="call", strike=K, qty=-1)],
        "protective_put":   [dict(kind="stock", qty=1), dict(kind="put", strike=K, qty=1)],
        "bull_call_spread": [dict(kind="call", strike=K, qty=1), dict(kind="call", strike=K + w, qty=-1)],
        "bear_put_spread":  [dict(kind="put", strike=K, qty=1), dict(kind="put", strike=K - w, qty=-1)],
        "long_straddle":    [dict(kind="call", strike=K, qty=1), dict(kind="put", strike=K, qty=1)],
        "long_strangle":    [dict(kind="call", strike=K + w, qty=1), dict(kind="put", strike=K - w, qty=1)],
    }
    if name == "custom":
        return [dict(leg) for leg in CUSTOM_LEGS]
    check(name in presets, f"STRATEGY must be one of {list(presets) + ['custom']}")
    return presets[name]


def leg_premium(leg):
    if leg.get("premium") is not None:
        return float(leg["premium"]), "your input"
    if leg["kind"] == "stock":
        return S0, "spot"
    if (USE_MARKET_PRICE_FOR_MAIN_LEG and MARKET_PRICE is not None and leg["kind"] == OPTION_TYPE
            and float(leg["strike"]) == float(STRIKE)):
        return MARKET_PRICE, "market"
    return binomial_price(S_MODEL, float(leg["strike"]), T, R, Q, SIGMA, TREE_STEPS, leg["kind"], EXERCISE), "model"


LEGS = build_legs(STRATEGY)
for leg in LEGS:
    check(leg["kind"] in ("call", "put", "stock"), "each leg 'kind' must be call, put or stock")
    leg["prem"], leg["src"] = leg_premium(leg)

leg_rows = [(("Buy" if l["qty"] > 0 else "Sell") + f" {abs(l['qty'])}", l["kind"],
             "-" if l["kind"] == "stock" else f"{l['strike']:,.0f}", f"{l['prem']:,.2f}", l["src"]) for l in LEGS]
display(pd.DataFrame(leg_rows, columns=["Action (lots)", "Instrument", "Strike", "Price per share", "Price source"]))

strikes_used = [float(l["strike"]) for l in LEGS if l["kind"] != "stock"]
ST = np.unique(np.concatenate([np.linspace(0.6 * S0, 1.4 * S0, 401), strikes_used]))   # grid includes every strike
pnl_expiry = np.zeros_like(ST)
pnl_today = np.zeros_like(ST)
for l in LEGS:
    if l["kind"] == "stock":
        pay, now = ST, ST
    else:
        pay = _intrinsic(ST, float(l["strike"]), l["kind"])
        now = None
        if EXERCISE == "european":
            now = np.array([bs_price(s - PV_DIV, float(l["strike"]), T, R, Q, SIGMA, l["kind"]) for s in ST])
    pnl_expiry += l["qty"] * (pay - l["prem"])
    if pnl_today is not None and now is not None:
        pnl_today += l["qty"] * (now - l["prem"])
    else:
        pnl_today = None
pnl_expiry *= LOT_SIZE
if pnl_today is not None:
    pnl_today *= LOT_SIZE

net_cost = sum(l["qty"] * l["prem"] for l in LEGS) * LOT_SIZE
sgn = np.sign(pnl_expiry)
crossings = np.where(sgn[:-1] * sgn[1:] < 0)[0]
breakevens = [ST[i] - pnl_expiry[i] * (ST[i + 1] - ST[i]) / (pnl_expiry[i + 1] - pnl_expiry[i]) for i in crossings]
slope_end = pnl_expiry[-1] - pnl_expiry[-2]

print(f"Cost to open (per lot): Rs {net_cost:,.0f}  ({'you pay' if net_cost > 0 else 'you receive'})")
print(f"Best case in chart range : Rs {pnl_expiry.max():,.0f}" + ("  (keeps rising above the chart: effectively unlimited)" if slope_end > 1e-9 else ""))
print(f"Worst case in chart range: Rs {pnl_expiry.min():,.0f}" + ("  (keeps falling above the chart: unlimited loss)" if slope_end < -1e-9 else ""))
print("Break-even at expiry     :", ", ".join(f"Rs {b:,.0f}" for b in breakevens) if breakevens else "none inside the chart range")

plt.figure()
plt.plot(ST, pnl_expiry, color="black", label="at expiry")
if pnl_today is not None:
    plt.plot(ST, pnl_today, color="tab:blue", ls="--", label="today (model)")
plt.fill_between(ST, pnl_expiry, 0, where=pnl_expiry >= 0, color="green", alpha=0.2)
plt.fill_between(ST, pnl_expiry, 0, where=pnl_expiry < 0, color="red", alpha=0.2)
plt.axhline(0, color="grey", lw=1)
plt.axvline(S0, color="grey", ls=":", lw=1)
for b in breakevens:
    plt.scatter([b], [0], color="black", zorder=5)
plt.xlabel("stock price at expiry (Rs)")
plt.ylabel(f"profit / loss per lot (Rs), lot = {LOT_SIZE}")
plt.title(STRATEGY.replace("_", " ").title())
plt.legend()
plt.tight_layout()
plt.show()

## 12. Interactive explorer
Move the sliders and watch the price, Greeks and chart change. Needs `ipywidgets` (`pip install ipywidgets`).
The explorer ignores discrete dividends (it uses the dividend yield only).

In [ ]:
def explore(spot=S0, strike=STRIKE, vol_pct=SIGMA * 100, rate_pct=R * 100, days=DAYS,
            option=OPTION_TYPE, exercise=EXERCISE, steps=min(TREE_STEPS, 400)):
    T_ = max(int(days), 1) / 365.0
    price = binomial_price(spot, strike, T_, rate_pct / 100, Q, vol_pct / 100, int(steps), option, exercise)
    g = greeks(spot, strike, T_, rate_pct / 100, Q, vol_pct / 100, option, exercise)
    print(f"{exercise.title()} {option}: price Rs {price:,.2f} per share (Rs {price * LOT_SIZE:,.0f} per lot)   |   "
          f"delta {g['delta']:.3f}  gamma {g['gamma']:.5f}  vega {g['vega']:.2f}  theta {g['theta']:.2f}/day  rho {g['rho']:.2f}")
    xs = np.linspace(0.7 * spot, 1.3 * spot, 80)
    now = [fast_price(x, strike, T_, rate_pct / 100, Q, vol_pct / 100, option, exercise) for x in xs]
    plt.figure(figsize=(8, 4))
    plt.plot(xs, now, label="value today")
    plt.plot(xs, _intrinsic(xs, strike, option), "--", label="value at expiry")
    plt.scatter([spot], [price], color="black", zorder=5)
    plt.axvline(strike, color="grey", ls=":", lw=1)
    plt.xlabel("stock price (Rs)")
    plt.ylabel("option value per share (Rs)")
    plt.legend()
    plt.tight_layout()
    plt.show()


try:
    import ipywidgets as widgets
    _w = dict(style={"description_width": "110px"}, layout=widgets.Layout(width="480px"))
    widgets.interact(
        explore,
        spot=widgets.FloatSlider(value=S0, min=round(0.7 * S0), max=round(1.3 * S0), step=1.0, description="Stock price", **_w),
        strike=widgets.FloatSlider(value=STRIKE, min=round(0.7 * S0), max=round(1.3 * S0), step=5.0, description="Strike", **_w),
        vol_pct=widgets.FloatSlider(value=round(SIGMA * 100, 1), min=5, max=80, step=0.5, description="Volatility %", **_w),
        rate_pct=widgets.FloatSlider(value=round(R * 100, 2), min=0, max=15, step=0.05, description="Interest rate %", **_w),
        days=widgets.IntSlider(value=max(DAYS, 1), min=1, max=365, step=1, description="Days to expiry", **_w),
        option=widgets.Dropdown(options=["call", "put"], value=OPTION_TYPE, description="Type", **_w),
        exercise=widgets.Dropdown(options=["european", "american"], value=EXERCISE, description="Exercise", **_w),
        steps=widgets.IntSlider(value=min(TREE_STEPS, 400), min=50, max=1000, step=50, description="Tree steps", **_w),
    )
except ImportError:
    print("ipywidgets is not installed, so the sliders are skipped.  Install with: pip install ipywidgets")
    print("You can still call explore(spot=1250, vol_pct=25) by hand.")

## 13. Things to double-check before relying on a number
The cell below lists every warning raised during this run.

In [ ]:
if WARNINGS:
    print("Warnings from this run:")
    for w in WARNINGS:
        print(" -", w)
else:
    print("No warnings raised during this run.")
print("""
Reality checks that no model can fix for you:
 - Volatility from the past is a guess about the future. Big news (results, budget, oil shocks) breaks it.
 - Real option prices have a volatility 'smile': different strikes trade at different implied volatilities.
   This notebook uses one flat volatility per option, so far-away strikes will show gaps against the market.
 - Check the strike, expiry date and lot size on your broker screen. Exchanges revise lot sizes.
 - Costs matter: brokerage, STT, exchange charges, taxes and the bid-ask gap can erase a small edge.
 - NSE stock options settle by physical delivery. In-the-money positions at expiry mean shares change hands.
 - This is an educational tool, not investment advice.
""")